# **Classification**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trendinafrica/TReND-CaMinA/blob/main/notebooks/Kenya26/10-Classification/classification.ipynb)

<img align="left" width="100" src="https://raw.githubusercontent.com/trendinafrica/TReND-CaMinA/main/images/CaMinA_logo.png">

### **TReND-CaMinA: Computational Neuroscience and Machine Learning Summer School, Kenya 2026**
#### **Content creator:** Evren Gokcen, Bosch Research

---

## **Overview**
In this lesson, we will learn about **classification** methods. Classification methods are used to predict discrete labels (or classes) from data. Classification is a fundamental task in machine learning and is widely used in neuroscience to decode brain activity patterns.

In this notebook, you will decode intended arm-reach direction from the activity of a population of motor cortex neurons. We will build on the previous session, where you explored real recordings from dorsal premotor cortex (area PMd). Here we will use a **simulated** version of that reaching task, which lets us illustrate classification cleanly.

<center><img src="./logistic_regression_animation1D.gif" width=500><img src="./logistic_regression_animation2D.gif" width=500></center>

## **Learning objectives / key questions**
1. What is **classification**, and where does it apply in neuroscience?
2. Why is it possible to classify (decode) external variables (like movement direction, or visual stimulus identity) from neural activity?
3. What is the **Nearest Neighbors** method, and how do I implement it for one- and two-dimensional data?
4. What are **decision boundaries** and **decision regions**, and how are they used in classification?
5. What is the difference between **parametric** and **non-parametric** classification? And what are the strengths and limitations of each?
6. What does it mean to **tune** classifier **parameters**, and what are the shortcomings of tuning parameters by hand?

## **Contents**

**Core topics:**

0. [Import dependencies and data](#dependencies)
1. [Introduction: Decoding movement intention in the motor system](#introduction)
2. [Nearest neighbors classification and decision boundaries](#nearest-neighbors)
3. [Defining and tuning decision boundaries with parameters](#parametric-boundaries)

**Bonus topics:**

1. [Logistic regression](#logistic-regression)
2. [Population decoding](#population-decoding)
3. [Combining classification and PCA](#class-pca)
4. [Multiclass classification](#multiclass)
5. [Further reading](#further-reading)

## 0. Import dependencies and data <a name="dependencies"></a>

In [ ]:
# If running in Colab, mount the drive folder
COLAB = False
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
%pip install numpy scipy matplotlib ipywidgets plotly

import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
import scipy.io as spio

from collections import Counter
from ipywidgets import interact
from matplotlib.animation import FuncAnimation
from matplotlib.colors import to_hex
import plotly.graph_objects as go

## 1. Introduction: Decoding movement intention in the motor system <a name="introduction"></a>

Neurons in motor cortex carry information about movement, both *during* a reach and while it is being *planned*. In the previous session, you saw that planning-period activity recorded in dorsal premotor cortex (area **PMd**), while a monkey reached to one of several targets, is strongly structured by reach direction. Your PCA made this structure visible.

That structure is what lets us **decode** the intended movement from the activity. To build intuition, here is a *simulated* center-out reaching task. The population activity on the left is read out, in real time, to drive the cursor to each target on the right.

<center><img src="./reach_raster_animation.gif" width=680></center>

For the rest of this lesson, we'll take snapshots of this kind of activity, one per trial, and ask a **classification** question: from the activity alone, which reach direction was intended?

Let's start where we left off yesterday, with dimensionality reduction.
Today we'll work with a *simulated* center-out reaching dataset
(`reach_sim_data.mat`), designed to mimic the neural activity recorded in motor
cortex while an animal reaches to targets. It contains the variable:
- `Xplan`: a 2D array of shape `(n, d)`. Here we have `n = 800` trials and
`d = 10` simulated neurons. `Xplan[i, j]` is the activity (firing rate) of neuron
`j` on trial `i`, sampled in a planning window, just before the reach begins. There are 100 trials for each of 8
reaching angles, evenly spaced around the circle. Trials are ordered by angle, so
the first 100 trials are for angle 0°, the next 100 trials are for angle 45°, and
so on.

Because the data are simulated, the neurons have smooth, continuous-valued activity
and well-behaved tuning, which is convenient for illustrating classification without
the idiosyncrasies of real recordings.

### Revisiting PCA

#### Load the data

In [ ]:
# Set paths to data file
datapath  = ''  # add your path here
datafile  = 'reach_sim_data.mat'

# Load the Matlab format .mat data into Python
mat = spio.loadmat(datapath + datafile, squeeze_me=True)
Xplan = mat['Xplan']  # (n trials, d neurons), continuous-valued activity

# Check the shape of the data
n, d = Xplan.shape
print('Xplan dimensions: (n trials, d neurons)')
print((n, d))

# Basic constants (read from the dataset)
num_reaches_per_angle = int(mat['num_reaches_per_angle'])
num_angles = int(mat['num_angles'])
angles = [int(round(angle)) for angle in mat['angles']]  # degrees: 0, 45, ..., 315

# Train/test split settings used throughout (a 50/50 split with a fixed seed)
train_frac = float(mat['train_fraction'])
split_seed = int(mat['split_seed'])

# Neurons used in the classification demos below
demo_neuron = int(mat['demo_neuron'])
demo_neuron_pair = list(mat['demo_neuron_pair'])

#### Compute the sample covariance and diagonalize it (PCA)

Recall that the sample covariance matrix
$S = \frac{1}{n} \sum_i (\boldsymbol{x}_i-\boldsymbol{\mu})(\boldsymbol{x}_i-\boldsymbol{\mu})^{\intercal}$,
and the sample mean $\boldsymbol{\mu} = \frac{1}{n} \sum_i \boldsymbol{x}_i$.

In [ ]:
# First find the mean over trials (axis=0) of each neuron
mu = np.mean(Xplan, axis=0)

# Now compute the data covariance
Xplan_centered = Xplan - mu  # Don't forget to center your data first!
cov = (1/n) * Xplan_centered.T @ Xplan_centered  # Now you can compute the covariance

# Perform eigendecomposition of the data covariance
D, U = np.linalg.eig(cov)  # D = vector of eigenvalues, U = matrix of eigenvectors

# Make sure the eigenvalues are sorted (in descending order)
idx = np.argsort(D)[::-1]
D = D[idx]

# Arrange the eigenvectors according to the magnitude of the eigenvalues
U = U[:, idx]

#### Project the data onto the top 3 principal components and visualize

Recall that projections of the data points $\boldsymbol{x}_i \in \mathbb{R}^d$ into the
three-dimensional PC space are given by the PC scores, $\boldsymbol{z}_i \in \mathbb{R}^3$, where $z_k^i = \boldsymbol{u}_k^{\intercal} (\boldsymbol{x}_i - \boldsymbol{\mu})$, $k=1,2,3$.

In [ ]:
# Project every trial onto the top 3 PCs and show the result as an
# interactive 3D scatter. Drag to rotate, scroll to zoom.

# Color encodes reach direction on a cyclic (hsv) map, so neighboring
# angles get neighboring hues and the ring structure is easy to see.
colors = [to_hex(plt.cm.hsv(k / num_angles)) for k in range(num_angles)]

fig = go.Figure()
for reach_idx in range(num_angles):
    # Trials for this reach angle (data is grouped angle-by-angle).
    trial_idxs = np.arange(num_reaches_per_angle) + reach_idx * num_reaches_per_angle
    Z = Xplan_centered[trial_idxs] @ U[:, :3]  # (num_reaches_per_angle, 3) PC scores
    fig.add_trace(
        go.Scatter3d(
            x=Z[:, 0],
            y=Z[:, 1],
            z=Z[:, 2],
            mode='markers',
            name=f'{angles[reach_idx]}°',
            marker=dict(size=3.5, opacity=0.8, color=colors[reach_idx]),
        )
    )

fig.update_layout(
    template='plotly_white',
    autosize=True,
    height=520,
    title='Reach data projected into three-dimensional PC space',
    legend=dict(title='Reach angle', itemsizing='constant'),
    scene=dict(
        xaxis_title='PC 1', yaxis_title='PC 2', zaxis_title='PC 3', aspectmode='data'
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

**_💬 Discussion questions:_**

1. What is striking about these projections?
2. What property of motor-cortical neurons (like those in area PMd) might lead to this pattern?

### Tuning curves

Let's take a step back from PCA and compute tuning curves for each neuron.
Recall that the tuning curve of a neuron is the average firing rate of the neuron (over
trials) as a function of the reach angle.

In [ ]:
# It's helpful here to reshape the data into a 3D array, with an extra axis for the
# reach angle
Xplan_angles = Xplan.reshape((num_angles, num_reaches_per_angle, d))

# Average firing rate for each neuron at each reach angle, and the standard error
# of the mean (SEM) across trials
tuning_curves = np.mean(Xplan_angles, axis=1)  # (num_angles, d)
sem_tuning_curves = np.std(Xplan_angles, axis=1) / np.sqrt(num_reaches_per_angle)

# The two neurons we'll use later for classification
top_neurons = np.array(demo_neuron_pair)

fig, ax = plt.subplots(figsize=(6, 3.5))
for neuron in top_neurons:
    tuning_curve = tuning_curves[:, neuron]
    sem = sem_tuning_curves[:, neuron]
    line, = ax.plot(
        angles, tuning_curve, marker='o', markersize=5, label=f'Neuron {neuron}'
    )
    # SEM band in the same color as its curve
    ax.fill_between(
        angles,
        tuning_curve - sem,
        tuning_curve + sem,
        color=line.get_color(),
        alpha=0.2,
    )

ax.set_xlabel('Reach angle (\u00b0)')
ax.set_ylabel('Average firing rate (Hz)')
ax.set_title('Example tuning curves')
ax.set_xticks(angles)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
plt.tight_layout()
plt.show()

**_💬 Discussion questions:_**

- What is the shape of the tuning curves? How does that relate to the nature of reach angles?

## 2. Nearest neighbors classification and decision boundaries <a name="nearest-neighbors"></a>

Given how nicely the neural responses appear to be organized according to reach angle,
both based on PCA and tuning curves, perhaps we can use this information to
decode, or classify, the reach angle from the neural responses.

We'll start our journey with the **Nearest Neighbors** classifier.

### Binary classification with one neuron

To warm up, let's simplify the problem first:
1. Let's consider only two reach angles. We are thus interested in
   *binary classification*.
2. Let's consider only one neuron's response, and see how far we can get in decoding.

In [ ]:
neuron_idx = demo_neuron  # The neuron we'll use for binary classification
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, neuron_idx]
X_reach2 = Xplan_angles[reach_idx2, :, neuron_idx]

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])  # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]

# Visualize each trial's response as a point at y=0 (0° reach) or y=1 (45° reach),
# the same layout we'll use later for logistic regression, with the
# per-class firing-rate histograms above, sharing the firing-rate axis.
reach1_color, reach2_color = 'tab:red', 'tab:blue'
fig, (ax_hist, ax_pts) = plt.subplots(
    2, 1, figsize=(7, 4.6), sharex=True,
    gridspec_kw={'height_ratios': [1, 1.3], 'hspace': 0.08},
)

# Top: class-conditional histograms on a shared set of bins.
lo = min(X_train_reach1.min(), X_train_reach2.min())
hi = max(X_train_reach1.max(), X_train_reach2.max())
bins = np.linspace(lo, hi, 16)
ax_hist.hist(
    X_train_reach1,
    bins=bins,
    color=reach1_color,
    alpha=0.55,
    label=f'{angles[reach_idx1]}° reach',
)
ax_hist.hist(
    X_train_reach2,
    bins=bins,
    color=reach2_color,
    alpha=0.55,
    label=f'{angles[reach_idx2]}° reach',
)
ax_hist.set_ylabel('Count')
ax_hist.legend(frameon=False, loc='upper right')
ax_hist.spines[['top', 'right']].set_visible(False)

# Bottom: each trial as a point at its class height.
ax_pts.plot(
    X_train_reach1,
    np.zeros_like(X_train_reach1),
    'o',
    color=reach1_color,
    alpha=0.5,
    markersize=6,
)
ax_pts.plot(
    X_train_reach2,
    np.ones_like(X_train_reach2),
    'o',
    color=reach2_color,
    alpha=0.5,
    markersize=6,
)
ax_pts.set_yticks([0, 1])
ax_pts.set_yticklabels([f'{angles[reach_idx1]}° reach', f'{angles[reach_idx2]}° reach'])
ax_pts.set_ylim(-0.4, 1.4)
ax_pts.set_xlabel('Firing rate (Hz)')
ax_pts.spines[['top', 'right']].set_visible(False)

fig.suptitle(f'Neuron {neuron_idx} responses for two reach angles', y=0.98)
plt.tight_layout()
plt.show()

It looks like these neural responses have decent separation by the two angles!

What if we bring in a few test points?

In [ ]:
# Two "anonymous" test points near each extreme and two in the ambiguous middle.
reach1_tests = [38, 4]  # true 0° trials: one clear, one near the boundary
reach2_tests = [16, 10]  # true 45° trials: one clear, one near the boundary

# Training data at its class height, with unlabeled test points (black stars)
# dropped in between. Where would you assign each one?
reach1_color, reach2_color = 'tab:red', 'tab:blue'
fig, ax = plt.subplots(figsize=(7, 3.2))
# Training points
ax.plot(
    X_train_reach1,
    np.zeros_like(X_train_reach1),
    'o',
    color=reach1_color,
    alpha=0.5,
    markersize=6,
)
ax.plot(
    X_train_reach2,
    np.ones_like(X_train_reach2),
    'o',
    color=reach2_color,
    alpha=0.5,
    markersize=6,
)
# Test points
for j, t in enumerate(reach1_tests):
    ax.plot(
        X_test_reach1[t],
        0.5,
        '*',
        color='k',
        markersize=14,
        label='test point' if j == 0 else None,
    )
for t in reach2_tests:
    ax.plot(
        X_test_reach2[t],
        0.5,
        '*',
        color='k',
        markersize=14,
    )
ax.set_yticks([0, 1])
ax.set_yticklabels([f'{angles[reach_idx1]}° reach', f'{angles[reach_idx2]}° reach'])
ax.set_ylim(-0.4, 1.4)
ax.set_xlabel('Firing rate (Hz)')
ax.set_title(f'Where do these test points belong? (Neuron {neuron_idx})')
ax.legend(frameon=False, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**_💬 Discussion questions:_**

- How would you classify each of these test points? Why?

### Nearest neighbors classification: A simple majority vote

We can formalize some of the intuitive principles you likely used above to get the
Nearest Neighbors classifier. In essence, the Nearest Neighbors classifier works
as follows: given a test data point, find the *k* training data points that are closest
to it, and those neighbors "choose" the class label of the test point by majority vote.

We'll then need three key components to implement this:

1. **Distance Metric**: We need a way to measure the distance or similarity between data
   points.
2. **k-Nearest Neighbors**: For a given test instance, we need to identify the k
   training data points that are closest to it.
3. **Majority Vote**: We need to identify the most common class label among the k
   nearest neighbors and assign that label to the test instance.

**_📝 Exercise_**

Let's start by creating a base KNN class. Wherever you see a `TODO` (there are **three**), attempt to fill in that line of code.

You can find the solution below.

In [ ]:
class KNNBase:
    def __init__(self, k: int = 3):
        """
        Initialize the KNN classifier with one hyperparameter: k.

        Parameters
        ----------
        k : int
            The number of nearest neighbors to consider for classification. Default: 3.
        """
        self.k = k

    def fit(self, X: np.ndarray, y: np.ndarray):
        """
        Fit the model to the training data.

        For the KNN classifier, "fitting" simply stores the training data and labels.

        Parameters
        ----------
        X : numpy array, shape (n, d)
            The training data, with n samples and d features.
        y : numpy array, shape (n,)
            The labels corresponding to the training data.
        """
        self.X_train = []  # TODO: Remove the brackets and fill in your solution
        self.y_train = []  # TODO: Remove the brackets and fill in your solution

    def distance(self, x1: np.ndarray, x2: np.ndarray) -> float:
        """
        Compute the distance between two points.

        Parameters
        ----------
        x1 : numpy array
            The first point.
        x2 : numpy array
            The second point.

        Returns
        -------
        float
            The distance between the two points.
        """
        # We'll implement this later
        pass

    def predict(self, x: np.ndarray) -> int:
        """
        Predict the labels for a single input data point.

        Parameters
        ----------
        x : numpy array, shape (d,)
            A single input data point for which to predict the label.

        Returns
        -------
        y_pred : int
            The predicted label for the input data point.
        """
        # Compute the distances from the input point to all training points
        distances = [self.distance(x, x_train) for x_train in self.X_train]

        # Get the closest k neighbors
        k_indices = []  # TODO - Hint: Use the np.argsort function (remove brackets)
        k_nearest_labels = [self.y_train[i] for i in k_indices]

        # Majority vote
        most_common = Counter(k_nearest_labels).most_common()
        return most_common[0][0]

    def predict_batch(self, X: np.ndarray) -> np.ndarray:
        """
        Predict the labels for a batch of input data points.

        Parameters
        ----------
        X : numpy array, shape (n, d)
            The input data for which to predict labels.

        Returns
        -------
        y_pred : numpy array, shape (n,)
            The predicted labels for the input data batch.
        """
        return np.array([self.predict(x) for x in X])

In [ ]:
#@title Double click to see solution {display-mode: "form"}
class KNNBase:
    def __init__(self, k: int = 3):
        """
        Initialize the KNN classifier with one hyperparameter: k.

        Parameters
        ----------
        k : int
            The number of nearest neighbors to consider for classification. Default: 3.
        """
        self.k = k

    def fit(self, X: np.ndarray, y: np.ndarray):
        """
        Fit the model to the training data.

        For the KNN classifier, "fitting" simply stores the training data and labels.

        Parameters
        ----------
        X : numpy array, shape (n, d)
            The training data, with n samples and d features.
        y : numpy array, shape (n,)
            The labels corresponding to the training data.
        """
        self.X_train = X  # SOLUTION
        self.y_train = y  # SOLUTION

    def distance(self, x1: np.ndarray, x2: np.ndarray) -> float:
        """
        Compute the distance between two points.

        Parameters
        ----------
        x1 : numpy array
            The first point.
        x2 : numpy array
            The second point.

        Returns
        -------
        float
            The distance between the two points.
        """
        pass

    def predict(self, x: np.ndarray) -> int:
        """
        Predict the labels for a single input data point.

        Parameters
        ----------
        x : numpy array, shape (d,)
            A single input data point for which to predict the label.

        Returns
        -------
        y_pred : int
            The predicted label for the input data point.
        """
        # Compute the distances from the input point to all training points
        distances = [self.distance(x, x_train) for x_train in self.X_train]

        # Get the closest k neighbors
        k_indices = np.argsort(distances)[:self.k]  # SOLUTION
        k_nearest_labels = [self.y_train[i] for i in k_indices]

        # Majority vote
        most_common = Counter(k_nearest_labels).most_common()
        return most_common[0][0]

    def predict_batch(self, X: np.ndarray) -> np.ndarray:
        """
        Predict the labels for a batch of input data points.

        Parameters
        ----------
        X : numpy array, shape (n, d)
            The input data for which to predict labels.

        Returns
        -------
        y_pred : numpy array, shape (n,)
            The predicted labels for the input data batch.
        """
        return np.array([self.predict(x) for x in X])

#### Distance Metric

We now have everything in place except for our distance metric. In our 1D
(single-neuron) case, we can simply use the absolute difference between the firing rate
of the neuron for the test point and the training points.

**_📝 Exercise_**

Implement the distance function below.

Wherever you see a `TODO` (there is **one**), attempt to fill in that line of code.

You can find the solution below.

In [ ]:
def abs_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """
    Compute the absolute distance between two 1D data points.

    Parameters
    ----------
    x1 : numpy array
        The first data point.
    x2 : numpy array
        The second data point.

    Returns
    -------
    float
        The absolute distance between the two points.
    """
    return None  # TODO: Replace None with your solution (use numpy functions)

In [ ]:
#@title Double click to see solution {display-mode: "form"}
def abs_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """
    Compute the absolute distance between two 1D data points.

    Parameters
    ----------
    x1 : numpy array
        The first data point.
    x2 : numpy array
        The second data point.

    Returns
    -------
    float
        The absolute distance between the two points.
    """
    return np.abs(x1 - x2)  # SOLUTION

Let's now use this distance metric to implement our first KNN classifier!

Instantiate the class:

In [ ]:
# To instantiate the class, all we need to provide is the number of neighbors k.
knn_1D = KNNBase(k=7)
# Then we can add the distance metric we just defined.
knn_1D.distance = abs_distance

**_💬 Discussion questions:_**

- Why might we choose to use $k=7$ neighbors for the vote, rather than, for example, $k=6$ neighbors?

**_📝 Exercise_**

"Train" the model!

Wherever you see a `TODO` (there is **one**), attempt to fill in that line of code.

You can find the solution below.

In [ ]:
# Prep the data for compatibility with our KNN class
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Fit the KNN model
# TODO

In [ ]:
#@title Double click to see solution {display-mode: "form" }
# Prep the data for compatibility with our KNN class
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Fit the KNN model
knn_1D.fit(X_train, y_train)  # SOLUTION

Having trained the model, we can now use it to make predictions on new data.
And importantly, we'll need to be able to **evaluate** how well the model performs on
these new unseen data points.

**_📝 Exercise_**

Implement the function to compute classification accuracy.

Wherever you see a `TODO` (there is **one**), attempt to fill in that line of code.

You can find the solution below.

In [ ]:
# Let's define a function to compute the accuracy of our KNN model
def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Compute the classification accuracy of the model.

    Parameters
    ----------
    y_true : numpy array, shape (n,)
        The true labels for the data points.
    y_pred : numpy array, shape (n,)
        The predicted labels for the data points.

    Returns
    -------
    float
        The classification accuracy, defined as the proportion of correct predictions.
    """
    return None  # TODO: Replace None with your solution (use numpy functions)

In [ ]:
#@title Double click to see solution {display-mode: "form"}
# Let's define a function to compute the accuracy of our KNN model
def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Compute the classification accuracy of the model.

    Parameters
    ----------
    y_true : numpy array, shape (n,)
        The true labels for the data points.
    y_pred : numpy array, shape (n,)
        The predicted labels for the data points.

    Returns
    -------
    float
        The classification accuracy, defined as the proportion of correct predictions.
    """
    return np.mean(y_true == y_pred)  # SOLUTION

Predict labels for the test data and evaluate our model's accuracy:

In [ ]:
# Prep the data for compatibility with our KNN class
# Train
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

# Predict the labels for the train data
y_pred_train = knn_1D.predict_batch(X_train)

# Predict the labels for the test data
y_pred_test = knn_1D.predict_batch(X_test)

# Train accuracy
acc_train = accuracy(y_train, y_pred_train)
print(f'Train accuracy: {acc_train:.3f}')

# Test accuracy
acc_test = accuracy(y_test, y_pred_test)
print(f'Test accuracy: {acc_test:.3f}')

That's not bad! Let's visualize the test points from before, and color them by
their predicted class labels. Do these predictions make sense?

In [ ]:
reach1_tests = [38, 4]  # true 0° trials
reach2_tests = [16, 10]  # true 45° trials

# Reshape the predicted test labels to include an additional axis for reach angle
y_pred_reaches = y_pred_test.reshape(2, -1).T
y_pred_reach1 = y_pred_reaches[:, 0]
y_pred_reach2 = y_pred_reaches[:, 1]

# Each test point is drawn at the height of its TRUE class (its row) and colored
# by the KNN PREDICTION. A star whose color matches its row was classified
# correctly; a mismatched color -- a red star up on the 45° row, or a blue star
# down on the 0° row -- is a misclassification.
reach1_color, reach2_color = 'tab:red', 'tab:blue'
pred_color = [reach1_color, reach2_color]  # index by predicted class (0 or 1)

fig, ax = plt.subplots(figsize=(7, 3.2))
# Training points
ax.plot(
    X_train_reach1,
    np.zeros_like(X_train_reach1),
    'o',
    color=reach1_color,
    alpha=0.5,
    markersize=6,
)
ax.plot(
    X_train_reach2,
    np.ones_like(X_train_reach2),
    'o',
    color=reach2_color,
    alpha=0.5,
    markersize=6,
)
# Test points
for t in reach1_tests:
    ax.plot(
        X_test_reach1[t],
        0,
        '*',
        color=pred_color[int(y_pred_reach1[t])],
        markersize=16,
        markeredgecolor='k',
    )
for t in reach2_tests:
    ax.plot(
        X_test_reach2[t],
        1,
        '*',
        color=pred_color[int(y_pred_reach2[t])],
        markersize=16, markeredgecolor='k',
    )

# Legend proxies: star color encodes the prediction.
ax.scatter([], [], marker='*', color=reach1_color, edgecolor='k', s=200,
           label=f'predicted {angles[reach_idx1]}°')
ax.scatter([], [], marker='*', color=reach2_color, edgecolor='k', s=200,
           label=f'predicted {angles[reach_idx2]}°')
ax.set_yticks([0, 1])
ax.set_yticklabels([f'{angles[reach_idx1]}° reach', f'{angles[reach_idx2]}° reach'])
ax.set_ylim(-0.4, 1.4)
ax.set_xlabel('Firing rate (Hz)')
ax.set_title('Test predictions: row = true class, star color = KNN prediction')
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**_💬 Discussion questions:_**

- Based on the predicted labels shown above, is there a simple rule that
   would allow us to classify the test points correctly, without using the KNN
   classifier?

### Decision boundaries, part 1

It seems like there might be a critical threshold firing rate: Below this threshold,
the model predicts one angle, and above it, the other angle.

This introduces the concept of a **decision boundary**. Let's visualize our KNN
classifier's **decision regions**, and the corresponding decision boundary that results
from them.

In [ ]:
# Visualize the decision boundary: predict the class across a fine grid of
# firing rates and shade each predicted-class region. Training points sit on
# top: a point lying in the other class's shaded region is one this KNN would
# get wrong.

# Note: Depending on the value of k, the boundary can be slightly ragged: unlike a
# single threshold, KNN's prediction can flip back and forth where the two classes
# overlap.
x_min, x_max = X_train.min() - 0.5, X_train.max() + 0.5
xx = np.linspace(x_min, x_max, 400)
y_grid_pred = np.array(knn_1D.predict_batch(xx))

reach1_color, reach2_color = 'tab:red', 'tab:blue'
fig, ax = plt.subplots(figsize=(7, 3.2))

# Shade contiguous predicted-class regions.
edges = np.concatenate([[xx[0]], xx[np.where(np.diff(y_grid_pred) != 0)[0]], [xx[-1]]])
for a, b in zip(edges[:-1], edges[1:]):
    cls = int(y_grid_pred[np.searchsorted(xx, (a + b) / 2)])
    ax.axvspan(a, b, color=(reach1_color if cls == 0 else reach2_color), alpha=0.15, lw=0)

# Training data at its class height.
ax.plot(
    X_train_reach1,
    np.zeros_like(X_train_reach1),
    'o',
    color=reach1_color,
    alpha=0.6,
    markersize=6,
)
ax.plot(
    X_train_reach2,
    np.ones_like(X_train_reach2),
    'o',
    color=reach2_color,
    alpha=0.6,
    markersize=6,
)

# Label the two predicted regions.
boundary = np.median(edges[1:-1]) if len(edges) > 2 else 0.5 * (x_min + x_max)
ax.text(0.5 * (x_min + boundary), 1.3, f'predicted {angles[reach_idx1]}°',
        color=reach1_color, ha='center', fontsize=9)
ax.text(0.5 * (boundary + x_max), 1.3, f'predicted {angles[reach_idx2]}°',
        color=reach2_color, ha='center', fontsize=9)
ax.set_yticks([0, 1])
ax.set_yticklabels([f'{angles[reach_idx1]}° reach', f'{angles[reach_idx2]}° reach'])
ax.set_ylim(-0.4, 1.5)
ax.set_xlim(x_min, x_max)
ax.set_xlabel('Firing rate (Hz)')
ax.set_title('Decision boundary of the 1D KNN model')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**_💬 Discussion questions:_**

- Does this decision boundary align with our test predictions, above?

### Binary classification with two neurons

We're now ready to extend our Nearest Neighbors classifier to two neurons!

In [ ]:
# Let's start by visualizing the training data for two neurons
neuron_idxs = demo_neuron_pair  # The neurons we'll use for binary classification
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, neuron_idxs].T
X_reach2 = Xplan_angles[reach_idx2, :, neuron_idxs].T

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])  # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]

# Visualize the responses on each trial in the 2-neuron feature space.
reach1_color, reach2_color = 'tab:red', 'tab:blue'
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(
    X_train_reach1[:, 0],
    X_train_reach1[:, 1],
    color=reach1_color,
    alpha=0.6,
    s=36,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx1]}° reach',
)
ax.scatter(
    X_train_reach2[:, 0],
    X_train_reach2[:, 1],
    color=reach2_color,
    alpha=0.6,
    s=36,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx2]}° reach',
)
ax.set_xlabel(f'Neuron {neuron_idxs[0]} firing rate (Hz)')
ax.set_ylabel(f'Neuron {neuron_idxs[1]} firing rate (Hz)')
ax.set_title(f'Responses for two reach angles, Neurons {neuron_idxs[0]} and {neuron_idxs[1]}')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

This looks reasonably separable! How should we modify our KNN class to handle two neurons?

**_📝 Exercise_**

Implement the function to compute Euclidean distance between two points.

Wherever you see a `TODO` (there is **one**), attempt to fill in that line of code.

You can find the solution below.

In [ ]:
def euclidean_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """
    Compute the Euclidean distance between two data points.

    Parameters
    ----------
    x1 : numpy array
        The first data point.
    x2 : numpy array
        The second data point.

    Returns
    -------
    float
        The Euclidean distance between the two points.
    """
    return None  # TODO: Replace None with your solution (use numpy functions)

In [ ]:
#@title Double click to see solution {display-mode: "form"}
def euclidean_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """
    Compute the Euclidean distance between two data points.

    Parameters
    ----------
    x1 : numpy array
        The first data point.
    x2 : numpy array
        The second data point.

    Returns
    -------
    float
        The Euclidean distance between the two points.
    """
    return np.sqrt(np.sum((x1 - x2) ** 2))  # SOLUTION

We can now take the same steps as before to create and train our KNN classifier.

Instantiate the class:

In [ ]:
knn_2D = KNNBase(k=7)
knn_2D.distance = euclidean_distance

"Train" the model:

In [ ]:
# Prep the data for compatibility with our KNN class
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Fit the KNN model
knn_2D.fit(X_train, y_train)

Predict labels for the test data and evaluate our model's accuracy:

In [ ]:
# Prep the data for compatibility with our KNN class
# Train
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

# Predict the labels for the train data
y_pred_train = knn_2D.predict_batch(X_train)

# Predict the labels for the test data
y_pred_test = knn_2D.predict_batch(X_test)

# Train accuracy
acc_train = accuracy(y_train, y_pred_train)
print(f'Train accuracy: {acc_train:.3f}')

# Test accuracy
acc_test = accuracy(y_test, y_pred_test)
print(f'Test accuracy: {acc_test:.3f}')

That's again quite good! And even better than single-neuron classifier!

### Decision boundaries, part 2

Let's revisit the idea of decision boundaries, but now for our two-dimensional data.

**_💬 Discussion questions:_**
- How would you draw the decision boundary for our two-neuron KNN
classifier?

Then let's go ahead and visualize the true decision boundary for our classifier.

In [ ]:
# Visualize the 2D decision boundary, and make k interactive. For each k we
# predict the class on a grid, shade each predicted-class region, and draw the
# KNN boundary on top. Training points lying in the other class's region are
# ones this model would misclassify. Slide k to see how it controls the
# boundary's complexity.
from IPython.display import display

reach1_color, reach2_color = 'tab:red', 'tab:blue'

# Build the figure, static training points, and axes ONCE. The grid depends only
# on the (fixed) training data, so build it once too; the slider just re-classifies
# it and swaps the shaded regions + boundary contour. A coarser grid than the
# static plot (70 vs 100) keeps the pure-Python KNN responsive on each update.
x_min, x_max = X_train[:, 0].min() - 0.2, X_train[:, 0].max() + 0.2
y_min, y_max = X_train[:, 1].min() - 0.2, X_train[:, 1].max() + 0.2
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 70), np.linspace(y_min, y_max, 70))
grid_points = np.c_[xx.ravel(), yy.ravel()]  # shape (4900, 2)

fig, ax = plt.subplots(figsize=(6, 5))
plt.close(fig)
ax.scatter(
    X_train_reach1[:, 0],
    X_train_reach1[:, 1],
    color=reach1_color,
    alpha=0.6,
    s=36,
    edgecolor='white',
    linewidth=0.4,
    zorder=2,
    label=f'{angles[reach_idx1]}° reach',
)
ax.scatter(
    X_train_reach2[:, 0],
    X_train_reach2[:, 1],
    color=reach2_color,
    alpha=0.6,
    s=36,
    edgecolor='white',
    linewidth=0.4,
    zorder=2,
    label=f'{angles[reach_idx2]}° reach',
)
ax.set_xlabel(f'Neuron {neuron_idxs[0]} firing rate (Hz)')
ax.set_ylabel(f'Neuron {neuron_idxs[1]} firing rate (Hz)')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.spines[['top', 'right']].set_visible(False)
region_artists = []  # contourf/contour sets; cleared and redrawn each update
out = widgets.Output()


def render(k):
    # Refit a fresh KNN for the chosen k, then classify the grid.
    knn = KNNBase(k=k)
    knn.distance = euclidean_distance
    knn.fit(X_train, y_train)
    y_grid_pred = np.array(knn.predict_batch(grid_points)).reshape(xx.shape)

    # Swap the shaded regions + boundary contour (drawn under the points).
    for artist in region_artists:
        artist.remove()
    region_artists.clear()
    region_artists.append(
        ax.contourf(
            xx,
            yy,
            y_grid_pred,
            levels=[-0.5, 0.5, 1.5],
            colors=[reach1_color, reach2_color],
            alpha=0.16,
            zorder=0,
        )
    )
    region_artists.append(
        ax.contour(
            xx,
            yy,
            y_grid_pred,
            levels=[0.5],
            colors='0.4',
            linewidths=1.0,
            zorder=1,
        )
    )

    acc_train = accuracy(y_train, knn.predict_batch(X_train))
    acc_test = accuracy(y_test, knn.predict_batch(X_test))
    ax.set_title(f'2D KNN, k = {k}    |    train acc {acc_train:.2f}, test acc {acc_test:.2f}')
    with out:
        out.clear_output(wait=True)
        display(fig)


# step=2 keeps k odd, so binary votes never tie. continuous_update=False so the
# (slow, pure-Python) grid classification only runs when the slider is released.
k_slider = widgets.IntSlider(min=1, max=25, step=2, value=7, description='k',
                             continuous_update=False)
k_slider.observe(lambda change: render(change['new']), names='value')
render(k_slider.value)
display(k_slider, out)

**_📝 Exercises_**

1. Use the slider above to try different values for $k$. What happens to the decision boundary as $k$ changes? How does $k$ control the effective complexity of the classifier? What happens to the train and test accuracy at very small and very large $k$?
2. Try different pairs of reach angles to classify. Changing the angle pair changes the data itself, so this one isn't a slider: edit `reach_idx1` and `reach_idx2` at the top of the *"Binary classification with two neurons"* cell above, then re-run the cells from there down to this plot. Which pairs give the best classification performance, and why? Which give the worst, and why?

## 3. Defining and tuning decision boundaries with parameters <a name="parametric-boundaries"></a>

You may have noticed that, for certain values of $k$, the decision boundary looks
close to a straight line. Perhaps we can do well, then, by directly choosing
the decision boundary to be linear, and then estimating the best slope and intercept
of that line from the data.

### One-dimensional decision boundaries: A simple threshold value

Let's return to the one-dimensional, single-neuron case. Again, we want to classify two reach angles, using the spike counts from one neuron.

This time, let's set a threshold value $b$ directly, with the following rule:
$$\text{If } x < b, \text{ then predict angle 1. Else, predict angle 2.}$$

Note: This threshold value $b$ is frequently referred to as the *bias* term in machine learning.

In [ ]:
#@title Let's prepare the 1D data as before {display-mode: "form"}
neuron_idx = demo_neuron  # The neuron we'll use for binary classification
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, neuron_idx]
X_reach2 = Xplan_angles[reach_idx2, :, neuron_idx]

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])  # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

Then let's play around with a threshold ourselves, and see how it affects the decision boundary and the classification performance.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

reach1_color, reach2_color = 'tab:red', 'tab:blue'

# Build the figure and static points ONCE; the slider only moves the threshold
# line, re-shades the two decision regions, and updates the title. Output is
# driven through an Output widget with clear_output(wait=True) to reduce the
# flicker of the inline backend swapping a new image each update.
fig, ax = plt.subplots(figsize=(7, 3.2))
plt.close(fig)
ax.plot(
    X_train_reach1,
    np.zeros_like(X_train_reach1),
    'o',
    color=reach1_color,
    alpha=0.6,
    markersize=6,
)
ax.plot(
    X_train_reach2,
    np.ones_like(X_train_reach2),
    'o',
    color=reach2_color,
    alpha=0.6,
    markersize=6,
)

x_min, x_max = X_train.min() - 0.3, X_train.max() + 0.3
ax.set_xlim(x_min, x_max)
ax.set_ylim(-0.4, 1.5)
ax.set_yticks([0, 1])
ax.set_yticklabels([f'{angles[reach_idx1]}° reach', f'{angles[reach_idx2]}° reach'])
ax.set_xlabel('Firing rate (Hz)')
ax.spines[['top', 'right']].set_visible(False)
# Static region labels (the threshold stays interior, so left is always 0 deg).
ax.text(x_min + 0.1, 1.32, f'predict {angles[reach_idx1]}°', color=reach1_color, fontsize=9, va='center')
ax.text(x_max - 0.1, 1.32, f'predict {angles[reach_idx2]}°', color=reach2_color, fontsize=9, va='center', ha='right')

threshold_line = ax.axvline(np.median(X_train), color='m', linestyle='--', lw=1.5)
region_fills = []
out = widgets.Output()


def render(b):
    threshold_line.set_xdata([b, b])
    # Shade the predicted regions: below b -> 0 deg (red), above b -> 45 deg (blue).
    for coll in region_fills:
        coll.remove()
    region_fills.clear()
    region_fills.append(ax.axvspan(x_min, b, color=reach1_color, alpha=0.10, lw=0))
    region_fills.append(ax.axvspan(b, x_max, color=reach2_color, alpha=0.10, lw=0))
    acc_train = accuracy(y_train, (X_train > b).astype(int))
    acc_test = accuracy(y_test, (X_test > b).astype(int))
    ax.set_title(f'Threshold b = {b:.2f}    |    train acc {acc_train:.2f}, test acc {acc_test:.2f}')
    with out:
        out.clear_output(wait=True)
        display(fig)


b_slider = widgets.FloatSlider(
    min=float(np.floor(X_train.min())), max=float(np.ceil(X_train.max())),
    step=0.01, value=float(np.round(np.median(X_train), 2)), description='b',
)
b_slider.observe(lambda change: render(change['new']), names='value')
render(b_slider.value)
display(b_slider, out)

**_💬 Discussion questions:_**

1. At which values of $b$ does this classifier perform as well as the KNN classifier (with $k = 7$)?
2. Are there values of $b$ where this classifier performs even better than the KNN classifier?

#### **Note:** Parametric vs non-parametric classifiers

This simple threshold classifier is our first example of a **parametric classifier**.
The parameter, in this case, is the threshold value $b$.

### Two-dimensional decision boundaries: A line with a slope and intercept

Let's now extend this idea to two dimensions. We can represent the decision boundary as a line with a slope and intercept:
$$y = wx + b$$

where $w$ is the slope and $b$ is the intercept, or bias. And from this line, we can define
the decision rule as follows:
$$\text{If } y > wx + b, \text{ then predict angle 1. Else, predict angle 2.}$$

In [ ]:
#@title Let's prepare the 2D data as before {display-mode: "form"}

neuron_idxs = demo_neuron_pair  # The neurons we'll use for binary classification
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, neuron_idxs].T
X_reach2 = Xplan_angles[reach_idx2, :, neuron_idxs].T

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])  # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

Then we can now play around with the slope $w$ and the intercept $b$, and see how they affect the decision boundary and the classification performance.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

reach1_color, reach2_color = 'tab:red', 'tab:blue'

# Build the figure and the static training scatter ONCE; the slider callback
# only moves the boundary line and re-shades the two half-planes. Output is
# driven through an Output widget with clear_output(wait=True) to reduce the
# flicker of the inline backend swapping a new image each update.
fig, ax = plt.subplots(figsize=(6.5, 5))
plt.close(fig)

ax.scatter(
    X_train_reach1[:, 0],
    X_train_reach1[:, 1],
    color=reach1_color,
    alpha=0.6,
    s=36,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx1]}° reach',
)
ax.scatter(
    X_train_reach2[:, 0],
    X_train_reach2[:, 1],
    color=reach2_color,
    alpha=0.6,
    s=36,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx2]}° reach',
)

# Fixed limits so the plot does not jump around as the line slope changes.
x_min, x_max = X_train[:, 0].min() - 0.2, X_train[:, 0].max() + 0.2
y_min, y_max = X_train[:, 1].min() - 0.2, X_train[:, 1].max() + 0.2
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel(f'Neuron {neuron_idxs[0]} firing rate (Hz)')
ax.set_ylabel(f'Neuron {neuron_idxs[1]} firing rate (Hz)')
ax.spines[['top', 'right']].set_visible(False)

xline = np.linspace(x_min, x_max, 100)
(boundary_line,) = ax.plot([], [], 'm--', lw=1.5, label='Decision boundary')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
region_fills = []  # shaded half-plane PolyCollections; cleared each update
out = widgets.Output()


def render(w, b):
    yline = w * xline + b
    boundary_line.set_data(xline, yline)
    # Re-shade the predicted half-planes. Rule: predict angle 1 above the line
    # (y > w*x + b), angle 2 below it.
    for coll in region_fills:
        coll.remove()
    region_fills.clear()
    yfill = np.clip(yline, y_min, y_max)
    region_fills.append(ax.fill_between(xline, yfill, y_max, color=reach1_color, alpha=0.10, lw=0))
    region_fills.append(ax.fill_between(xline, y_min, yfill, color=reach2_color, alpha=0.10, lw=0))
    y_pred_train = (X_train @ np.array([w, -1]) + b > 0).astype(int)
    y_pred_test = (X_test @ np.array([w, -1]) + b > 0).astype(int)
    acc_train = accuracy(y_train, y_pred_train)
    acc_test = accuracy(y_test, y_pred_test)
    ax.set_title(f'y = {w:.2f}·x + {b:.1f}    |    train acc {acc_train:.2f}, test acc {acc_test:.2f}')
    with out:
        out.clear_output(wait=True)
        display(fig)


w_slider = widgets.FloatSlider(min=-2, max=3, step=0.01, value=1.0, description='w')
b_slider = widgets.FloatSlider(min=-12, max=6, step=0.01, value=0.0, description='b')
w_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
b_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
render(w_slider.value, b_slider.value)
display(widgets.HBox([w_slider, b_slider]), out)

**_💬 Discussion questions:_**

1. How many parameters does our classifier now have?
2. At which values of the slope, $w$, and the intercept, $b$, does this classifier perform as well as the KNN classifier (with $k = 7$)?
3. Are there values of $w$ and $b$ where this classifier performs even better than the KNN classifier?

## **Wrapping up, and key concepts**

Congratulations on making it this far in the lesson! We covered a lot of ground!

**Key concepts covered**:

- We started by observing that the activity of motor cortex neurons is *highly
structured*, reliably reflecting the subject's *intended movement direction* on a
*trial-by-trial* basis.
- We then asked whether we can decode this intended movement direction for new, unseen
data points, using machine learning methods.
- We built and evaluated a **Nearest Neighbors** classifier, a fundamental
**non-parametric** classification method. Nearest Neighbors works from a simple but
powerful principle: new data points are assigned a class by **majority vote**.
- We saw that this majority vote principle creates an effective **decision boundary**:
data points on one side of the boundary are assigned one class; data points on the
other side of the boundary are assigned the other class.
- Importantly, the **complexity** of the decision boundary depends on the number of
neighbors used, $k$: larger $k$ produces simpler decision boundaries (even linear),
while smaller $k$ produces complex decision boundaries (sometimes quite exotic!) --
at the risk of **overfitting** to the training data and giving poor test performance, or
**generalization**.
- Finally, we saw that we could directly define a decision boundary with **parameters**,
leading to a **parametric** linear classifier that we tuned by hand.

**Looking forward**:

- In tomorrow's lesson, we'll cover **regression** (decoding *continuous* variables),
the complementary machine learning problem to classification (decoding *discrete*
variables).
- We'll also cover the key *improvement* over today's parametric approach: rather
than tuning the parameters by hand, can we tune them automatically? Stay tuned for more!
- The bonus topics below cover this key improvement, and others, in the context of a
fundamental parametric classification method, **Logistic Regression**. They also
generalize the concepts covered today to **more than two neurons** and
**more than two classes**. You are not required to read through these bonus topics,
but you are certainly encouraged to explore them after today and after the school!

# Bonus topics

## *Bonus 1*. Logistic regression <a name="logistic-regression"></a>

Our approach so far has been to manually set the slope and intercept of the decision boundary, and then to classify data points in a binary manner based on whether they fall above or below this line.

**_💬 Discussion questions:_**

What are the drawbacks of this approach? Consider the following prompts:

1. Should points very close to the decision boundary be treated the same as points far away from it?
2. What if the data are more complex? Especially, if the data are not clearly separable, and if we start considering more than two neurons?

### Probabilities and the logistic sigmoid function

To address the first limitation, we'll start working with probabilities.
Specifically, we want to be able to answer the following question:

*Given a data point, what is the probability that it belongs to class (reach angle) 1?*

In math terms, we would write this statement as:
$$P(C_1 | \mathbf{x})$$

Review question: What key properties define a probability?
- A probability is a value between 0 and 1.
- The sum of probabilities for all possible classes must equal 1.

### The logistic sigmoid function

This brings us to the logistic sigmoid function, which is defined as:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**_📝 Exercise_**

To gain intuition, let's implement the logistic sigmoid in code.

Wherever you see a `TODO` (there is **one**), attempt to fill in that line of code.

You can find the solution below.

In [ ]:
def sigmoid(z: float) -> float:
    """
    Compute the logistic sigmoid function for a given input z.

    Parameters
    ----------
    z : float
        The input value for which to compute the sigmoid.

    Returns
    -------
    float
        The output of the sigmoid function.
    """
    return None  # TODO: Replace None with your solution (use numpy functions)

In [ ]:
#@title Double click to see solution {display-mode: "form"}
def sigmoid(z: float) -> float:
    """
    Compute the logistic sigmoid function for a given input z.

    Parameters
    ----------
    z : float
        The input value for which to compute the sigmoid.

    Returns
    -------
    float
        The output of the sigmoid function.
    """
    return 1 / (1 + np.exp(-z))  # SOLUTION

In [ ]:
# Let's plot the sigmoid function for a range of z values
z_values = np.linspace(-10, 10, 200)
sigmoid_values = sigmoid(z_values)

fig, ax = plt.subplots(figsize=(7, 4))
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.plot(z_values, sigmoid_values, color='#5b3fa0', linewidth=2.2)
ax.set_xlabel('z')
ax.set_ylabel(r'$\sigma(z)$')
ax.set_title('The logistic sigmoid function')
ax.set_xlim(-10, 10)
ax.set_ylim(-0.05, 1.05)
ax.set_yticks([0, 0.5, 1])
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

This function is promising so far: for all input values $z$, it outputs a value between 0 and 1, which is exactly what we want for a probability.

But now, let's add another definition on top of $z$:
$$z = w \cdot x + b$$

So $z$ is now a linear function of an input variable $x$, with a weight parameter $w$ and a bias parameter $b$.

Let's now replot the logistic sigmoid function, but this time as a function of $x$,
rather than $z$. Specifically,
$$\sigma(w \cdot x + b) = \frac{1}{1 + e^{-(w \cdot x + b)}}$$

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Build the figure and static guides ONCE; the sliders only update the sigmoid
# curve, the 0.5-crossing line, and the title. Output via an Output widget with
# clear_output(wait=True) to reduce the inline backend's flicker.
x_vals = np.linspace(-10, 10, 200)
fig, ax = plt.subplots(figsize=(7, 4))
plt.close(fig)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
crossing_line = ax.axvline(0, color='gray', linestyle='--', linewidth=1)
(sigmoid_curve,) = ax.plot(x_vals, sigmoid(x_vals), color='#5b3fa0', linewidth=2.2)
ax.set_xlabel('x')
ax.set_ylabel(r'$\sigma(wx + b)$')
ax.set_xlim(-10, 10)
ax.set_ylim(-0.05, 1.05)
ax.set_yticks([0, 0.5, 1])
ax.spines[['top', 'right']].set_visible(False)
out = widgets.Output()


def render(w, b):
    sigmoid_curve.set_ydata(sigmoid(w * x_vals + b))
    # Vertical line where the sigmoid crosses 0.5 (z = 0, i.e. x = -b/w).
    if w != 0:
        crossing_line.set_xdata([-b / w, -b / w])
        crossing_line.set_visible(True)
    else:
        crossing_line.set_visible(False)
    ax.set_title(f'σ(wx + b):   w = {w:.2f},   b = {b:.1f}')
    with out:
        out.clear_output(wait=True)
        display(fig)


w_slider = widgets.FloatSlider(min=-5, max=5, step=0.05, value=1.0, description='w')
b_slider = widgets.FloatSlider(min=-5, max=5, step=0.5, value=0.0, description='b')
w_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
b_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
render(w_slider.value, b_slider.value)
display(widgets.HBox([w_slider, b_slider]), out)

**_💬 Discussion questions:_**

1. What effect does the weight $w$ have on the shape of the logistic sigmoid function? What happens when the magnitude of $w$ is very large or very small?
2. What effect does the bias $b$ have on the shape of the logistic sigmoid function?
3. What happens when $w$ becomes negative?

### Class probabilities

Here's the key point: as a function of our data points $x$, the logistic sigmoid
satisfies the key properties for defining probabilities. We will then define the probability of class 1 as follows:
$$P(C_1 | \mathbf{x}) = \sigma(w \cdot x + b) = \frac{1}{1 + e^{-(w \cdot x + b)}}$$

**_💬 Discussion questions:_**

How do we define the probability of class 2?

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# P(class 1) = sigmoid, P(class 2) = 1 - sigmoid. Build the figure and both
# curves ONCE; the sliders only update their heights, the 0.5-crossing line,
# and the title. Output via clear_output(wait=True) to reduce flicker.
class1_color, class2_color = 'tab:blue', 'tab:red'
x_vals = np.linspace(-10, 10, 200)

fig, ax = plt.subplots(figsize=(7, 4))
plt.close(fig)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
crossing_line = ax.axvline(0, color='gray', linestyle='--', linewidth=1)
(p1_curve,) = ax.plot(x_vals, sigmoid(x_vals), color=class1_color, linewidth=2.2,
                      label='P(class 1)')
(p2_curve,) = ax.plot(x_vals, 1 - sigmoid(x_vals), color=class2_color, linewidth=2.2,
                      label='P(class 2)')
ax.set_xlabel('x')
ax.set_ylabel('Class probability')
ax.set_xlim(-10, 10)
ax.set_ylim(-0.05, 1.05)
ax.set_yticks([0, 0.5, 1])
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.spines[['top', 'right']].set_visible(False)
out = widgets.Output()


def render(w, b):
    s = sigmoid(w * x_vals + b)
    p1_curve.set_ydata(s)
    p2_curve.set_ydata(1 - s)
    # Vertical line where the two probabilities cross 0.5 (x = -b/w).
    if w != 0:
        crossing_line.set_xdata([-b / w, -b / w])
        crossing_line.set_visible(True)
    else:
        crossing_line.set_visible(False)
    ax.set_title(f'Class probabilities:   w = {w:.2f},   b = {b:.1f}')
    with out:
        out.clear_output(wait=True)
        display(fig)


w_slider = widgets.FloatSlider(min=-5, max=5, step=0.05, value=1.0, description='w')
b_slider = widgets.FloatSlider(min=-5, max=5, step=0.5, value=0.0, description='b')
w_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
b_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
render(w_slider.value, b_slider.value)
display(widgets.HBox([w_slider, b_slider]), out)

### A logistic classifier

With the logistic sigmoid function, we now have a way to define the probability that a data point belongs to each class.

How do we use these probabilities to classify the data points? We can set a threshold on the probability:
$$\text{If } P(C_1 | \mathbf{x}) > 0.5, \text{ then predict class 1. Else, predict class 2.}$$

And now let's start applying this logistic sigmoid classifier to our data
points, starting with the 1D case:

In [ ]:
#@title Let's prepare the 1D data as before {display-mode: "form"}

neuron_idx = demo_neuron  # The neuron we'll use for binary classification
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, neuron_idx]
X_reach2 = Xplan_angles[reach_idx2, :, neuron_idx]

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])   # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

reach1_color, reach2_color, sigmoid_color = 'tab:red', 'tab:blue', '#5b3fa0'

# Static training points at y=0 (0°) and y=1 (45°); the sliders only update the
# sigmoid curve P(45°) = sigma(w*x + b), the 0.5-crossing line, and the title.
x_min, x_max = X_train.min() - 0.2, X_train.max() + 0.2
x_curve = np.linspace(x_min, x_max, 200)

fig, ax = plt.subplots(figsize=(7, 4))
plt.close(fig)
# class-probability heatmap
prob_img = ax.imshow(
    sigmoid(x_curve)[None, :],
    extent=[x_min, x_max, -0.1, 1.1],
    cmap='coolwarm_r',
    vmin=0,
    vmax=1,
    alpha=0.25,
    aspect='auto',
    origin='lower',
    zorder=0,
)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
crossing_line = ax.axvline(0, color='gray', linestyle='--', linewidth=1)
(sigmoid_curve,) = ax.plot(x_curve, sigmoid(x_curve), color=sigmoid_color, linewidth=2.2)
ax.plot(
    X_train_reach1,
    np.zeros_like(X_train_reach1),
    'o',
    color=reach1_color,
    alpha=0.7,
    markersize=6,
    markeredgecolor='white',
    markeredgewidth=0.4,
    label=f'{angles[reach_idx1]}° reach',
)
ax.plot(
    X_train_reach2,
    np.ones_like(X_train_reach2),
    'o',
    color=reach2_color,
    alpha=0.7,
    markersize=6,
    markeredgecolor='white',
    markeredgewidth=0.4,
    label=f'{angles[reach_idx2]}° reach',
)
ax.set_xlim(x_min, x_max)
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 0.5, 1])
ax.set_xlabel(f'Neuron {neuron_idx} firing rate (Hz)')
ax.set_ylabel(f'Probability of {angles[reach_idx2]}° reach')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.spines[['top', 'right']].set_visible(False)
out = widgets.Output()


def render(w, b):
    sigmoid_curve.set_ydata(sigmoid(w * x_curve + b))
    prob_img.set_data(sigmoid(w * x_curve + b)[None, :])
    # Decision boundary: where P(45°) crosses 0.5 (x = -b/w).
    if w != 0:
        crossing_line.set_xdata([-b / w, -b / w])
        crossing_line.set_visible(True)
    else:
        crossing_line.set_visible(False)
    y_pred_train = (sigmoid(w * X_train + b) > 0.5).astype(int)
    y_pred_test = (sigmoid(w * X_test + b) > 0.5).astype(int)
    acc_train = accuracy(y_train, y_pred_train)
    acc_test = accuracy(y_test, y_pred_test)
    ax.set_title(f'w = {w:.2f}, b = {b:.1f}    |    train acc {acc_train:.2f}, test acc {acc_test:.2f}')
    with out:
        out.clear_output(wait=True)
        display(fig)


w_slider = widgets.FloatSlider(min=3.0, max=5.0, step=0.01, value=3.2, description='w')
b_slider = widgets.FloatSlider(min=-25, max=-20, step=0.01, value=-21, description='b')
w_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
b_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
render(w_slider.value, b_slider.value)
display(widgets.HBox([w_slider, b_slider]), out)

**_💬 Discussion questions:_**

1. At which values of $w$ and $b$ does this classifier perform best? Is that performance the same or different from our simple threshold classifier before?
2. Is there a range of values for $w$ and $b$ where the classifier still performs well? If so, roughly, what are those ranges?
3. For one of the good classifiers you found, what is the probability of a 45 degree reach for a data point with a firing rate of the following values:
    - 5.25 Hz
    - 6.5 Hz
    - 5.75 Hz

  Think about the proportion of red points and blue points at each spike count value. How do the probabilities you found compare to the proportions of red and blue points at those spike counts?

### Two-dimensional logistic classifier

So how do we extend the logistic classifier to two dimensions? We can use the same
logistic sigmoid function, but now with two weights and a bias:
$$P(C_1 | \mathbf{x}) = \sigma(w_1 x_1 + w_2 x_2 + b) = \frac{1}{1 + e^{-(w_1 x_1 + w_2 x_2 + b)}}$$

We can also rewrite this expression with vectors as follows:
$$P(C_1 | \mathbf{x}) = \sigma(\mathbf{w}^{\intercal} \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^{\intercal} \mathbf{x} + b)}}$$
where $\mathbf{w} = [w_1, w_2]^{\intercal}$ is a vector of weights, and $\mathbf{x} = [x_1, x_2]^{\intercal}$ is a vector of input features.

Just as with the 1D case, we can define the probability of class 2 as:
$$P(C_2 | \mathbf{x}) = 1 - P(C_1 | \mathbf{x}) = 1 - \sigma(\mathbf{w}^{\intercal} \mathbf{x} + b)$$

And we can use the same thresholding rule to classify the data points:
$$\text{If } P(C_1 | \mathbf{x}) > 0.5, \text{ then predict class 1. Else, predict class 2.}$$

Let's then explore the behavior of this classifier on our 2D data.

In [ ]:
#@title Let's prepare the 2D data as before {display-mode: "form" }

neuron_idxs = demo_neuron_pair  # The neurons we'll use for binary classification
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, neuron_idxs].T
X_reach2 = Xplan_angles[reach_idx2, :, neuron_idxs].T

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])  # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

reach1_color, reach2_color = 'tab:red', 'tab:blue'

# Grid, heatmap, colorbar, and static points built ONCE. The sliders only update
# the P(45°) heatmap (imshow set_data), the 0.5 decision contour, and the title.
x_min, x_max = X_train[:, 0].min() - 0.2, X_train[:, 0].max() + 0.2
y_min, y_max = X_train[:, 1].min() - 0.2, X_train[:, 1].max() + 0.2
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))
grid = np.column_stack([xx.ravel(), yy.ravel()])

fig, ax = plt.subplots(figsize=(6.5, 5))
plt.close(fig)
# class-probability heatmap
im = ax.imshow(
    np.zeros_like(xx),
    origin='lower',
    extent=[x_min, x_max, y_min, y_max],
    cmap='coolwarm_r',
    vmin=0,
    vmax=1,
    alpha=0.75,
    aspect='auto',
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(f'P({angles[reach_idx2]}° reach)')
ax.scatter(
    X_train_reach1[:, 0],
    X_train_reach1[:, 1],
    color=reach1_color,
    alpha=0.7,
    s=32,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx1]}° reach',
)
ax.scatter(
    X_train_reach2[:, 0],
    X_train_reach2[:, 1],
    color=reach2_color,
    alpha=0.7,
    s=32,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx2]}° reach',
)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel(f'Neuron {neuron_idxs[0]} firing rate (Hz)')
ax.set_ylabel(f'Neuron {neuron_idxs[1]} firing rate (Hz)')
ax.legend(loc='upper left', framealpha=0.8, edgecolor='none')
ax.spines[['top', 'right']].set_visible(False)
contour_holder = [None]
out = widgets.Output()


def render(w1, w2, b):
    s = sigmoid(grid @ np.array([w1, w2]) + b).reshape(xx.shape)
    im.set_data(s)
    # 0.5 decision boundary contour (remove the previous one first).
    if contour_holder[0] is not None:
        contour_holder[0].remove()
    contour_holder[0] = ax.contour(xx, yy, s, levels=[0.5], colors='0.2', linewidths=1.2)
    y_pred_train = (sigmoid(X_train @ np.array([w1, w2]) + b) > 0.5).astype(int)
    y_pred_test = (sigmoid(X_test @ np.array([w1, w2]) + b) > 0.5).astype(int)
    acc_train = accuracy(y_train, y_pred_train)
    acc_test = accuracy(y_test, y_pred_test)
    ax.set_title(f'w1={w1:.2f}, w2={w2:.2f}, b={b:.1f}   |   train {acc_train:.2f}, test {acc_test:.2f}')
    with out:
        out.clear_output(wait=True)
        display(fig)


w1_slider = widgets.FloatSlider(min=-2, max=2, step=0.005, value=0.95, description='w1')
w2_slider = widgets.FloatSlider(min=-2, max=2, step=0.005, value=-0.75, description='w2')
b_slider = widgets.FloatSlider(min=-8, max=2, step=0.01, value=-1.5, description='b')
for _sl in (w1_slider, w2_slider, b_slider):
    _sl.observe(lambda change: render(w1_slider.value, w2_slider.value, b_slider.value), names='value')
render(w1_slider.value, w2_slider.value, b_slider.value)
display(widgets.HBox([w1_slider, w2_slider, b_slider]), out)

**_💬 Discussion questions:_**

1. Where is the decision boundary in the above plot? What is it's shape? At the boundary, which class has a higher probability?
2. At which values of $w_1$, $w_2$, and $b$ does this classifier perform best? Is that performance the same or different from our simple threshold classifier before?
3. Was it harder or easier to find good values for $w_1$, $w_2$, and $b$ than it was to find good values for $w$ and $b$ in the 1D case? Why?
4. Suppose we next used three neurons, instead of two neurons to classify these data. How many parameters will be in the three-neuron model? What if we used 100 neurons?

### Logistic regression: Automatically tuning the parameters of the logistic classifier

You may have noticed that it gets harder to find good values for the weight and
bias parameters as the data become more complex: either higher-dimensional, or more
difficult to separate by class. Can we instead automate the procedure for finding
good parameter values?

This brings us to the concept of *logistic regression*, a *supervised learning* method
that learns the parameters of a logistic classifier from labeled training data.

To train a logistic regression model, we need the following key components:
1. **Training Data**: A set of labeled data points, where each point has a feature
vector $\mathbf{x}$ and a corresponding class label $y$.
2. **Objective Function** or **Loss Function**: A function that quantifies how well the
model's predictions match the true labels.
3. **Optimization Algorithm**: An algorithm to minimize the loss function and find the
optimal parameters.

Then once we fit the model to the training data, we want to evaluate its performance
on unseen test data.

### The logistic regression objective function

**_💬 Discussion questions:_**

1. What is our objective with classification?
2. Could we use that objective to train a classification model? Why or why not?

We cannot use the classification accuracy directly as our objective function for
training a classification model because it is not a differentiable function. We
therefore cannot compute gradients with respect to the model parameters.

We then want an objective function that is smooth and differentiable, making it
suitable for optimization algorithms like gradient descent.

This is where probabilities again provide a significant advantage. We can use the
probabilities we defined earlier to create such an objective function.

Let's start by defining the likelihood of the data given the model parameters.
- Likelihood: The probability of observing the training data given the model parameters.
- The likelihood for a single data point is given by:
$$P(y_i | \mathbf{x}_i, \mathbf{w}, b) = P(C_1 | \mathbf{x}_i)^{y_i} P(C_2 | \mathbf{x}_i)^{1 - y_i}$$
where $y_i$ is the true class label for data point $\mathbf{x}_i$ (1 for class 1, 0 for class 2), and $P(C_1 | \mathbf{x}_i)$ is the predicted probability of class 1 given the input features $\mathbf{x}_i$.
- If we assume that the data points are independent, the likelihood of the entire dataset is the product of the individual likelihoods:
$$P(\mathbf{y} | \mathbf{X}, \mathbf{w}, b) = \prod_{i=1}^{n} P(y_i | \mathbf{x}_i, \mathbf{w}, b) = \prod_{i=1}^{n} P(C_1 | \mathbf{x}_i)^{y_i} P(C_2 | \mathbf{x}_i)^{1 - y_i}$$
where $\mathbf{y}$ is the vector of true class labels for all data points, $\mathbf{X}$ is the matrix of input features, and $n$ is the number of data points.

We seek a model that maximizes this likelihood function. This approach is known as *maximum likelihood estimation* (MLE). We could try to optimize this likelihood function directly. We will instead take its logarithm:
$$\log P(\mathbf{y} | \mathbf{X}, \mathbf{w}, b) = \sum_{i=1}^{n} \left[ y_i \log(P(C_1 | \mathbf{x}_i)) + (1 - y_i) \log(P(C_2 | \mathbf{x}_i)) \right]$$

We take the logarithm for two reasons:
1. It transforms the product into a sum, which is easier to work with mathematically and numerically.
2. The parameters that maximize the log-likelihood will also maximize the original likelihood function, so we can use the log-likelihood as our objective function, and arrive at the same set of parameters.

In the machine learning literature, the common practice is to *minimize* a *loss function*. As one final step, then, we can define the loss function as the negative log-likelihood, and also normalize by the number of training data points:
$$L(\mathbf{w}, b) = -\frac{1}{n} \sum_{i=1}^{n} \left[ (1 - y_i) \log(P(C_1 | \mathbf{x}_i)) + y_i \log(P(C_2 | \mathbf{x}_i)) \right]$$

This loss function is commonly known as the *cross-entropy loss* or *logistic loss*.

Recall that the predicted probabilities are given by the sigmoid function:
$$P(C_1 | \mathbf{x}_i) = \sigma(\mathbf{w}^{\intercal} \mathbf{x}_i + b)$$
$$P(C_2 | \mathbf{x}_i) = 1 - \sigma(\mathbf{w}^{\intercal} \mathbf{x}_i + b)$$

If we then plug in the expression for the predicted probabilities, we get, as a complete
expression for the loss function:
$$L(\mathbf{w}, b) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i  \cdot \log\left(\sigma(\mathbf{w}^{\intercal} \mathbf{x}_i + b)\right) + (1 - y_i) \cdot \log\left(1 - \sigma(\mathbf{w}^{\intercal} \mathbf{x}_i + b)\right) \right]$$

#### Logistic loss function intuition

In [ ]:
#@title Let's go back to our 1D data {display-mode: "form"}

neuron_idx = demo_neuron  # The neuron we'll use for binary classification
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, neuron_idx]
X_reach2 = Xplan_angles[reach_idx2, :, neuron_idx]

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])   # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

In [ ]:
# Let's implement the cross-entropy loss function
def cross_entropy_loss(y_true: np.ndarray, w: np.ndarray, b: float, X: np.ndarray) -> float:
    """
    Compute the cross-entropy loss for logistic regression.

    Parameters
    ----------
    y_true : numpy array, shape (n,)
        True labels (0 or 1).
    w : numpy array, shape (d,)
        Weights of the logistic regression model.
    b : float
        Bias term of the logistic regression model.
    X : numpy array, shape (n, d)
        Input features.

    Returns
    -------
    loss : float
        The cross-entropy loss.
    """
    # If w is a float, convert it to a 1D array
    if np.isscalar(w):
        w = np.array([w])
    if len(X.shape) == 1:
        X = X.reshape(-1, 1)
    z = X @ w + b
    p = sigmoid(z)
    # Clip for numerical stability
    p = np.clip(p, 1e-12, 1 - 1e-12)
    # Compute the cross-entropy loss
    loss = -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    return loss

In [ ]:
import ipywidgets as widgets
from IPython.display import display

reach1_color, reach2_color, sigmoid_color = 'tab:red', 'tab:blue', '#5b3fa0'

# Same sigmoid-on-data plot as before, now also reporting the log-likelihood
# (= -cross-entropy loss) so we can watch the loss respond to w and b.
x_min, x_max = X_train.min() - 0.2, X_train.max() + 0.2
x_curve = np.linspace(x_min, x_max, 200)

fig, ax = plt.subplots(figsize=(7, 4))
plt.close(fig)
# class-probability heatmap
prob_img = ax.imshow(
    sigmoid(x_curve)[None, :],
    extent=[x_min, x_max, -0.1, 1.1],
    cmap='coolwarm_r',
    vmin=0,
    vmax=1,
    alpha=0.25,
    aspect='auto',
    origin='lower',
    zorder=0,
)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
crossing_line = ax.axvline(0, color='gray', linestyle='--', linewidth=1)
(sigmoid_curve,) = ax.plot(x_curve, sigmoid(x_curve), color=sigmoid_color, linewidth=2.2)
ax.plot(
    X_train_reach1,
    np.zeros_like(X_train_reach1),
    'o',
    color=reach1_color,
    alpha=0.7,
    markersize=6,
    markeredgecolor='white',
    markeredgewidth=0.4,
    label=f'{angles[reach_idx1]}° reach',
)
ax.plot(
    X_train_reach2,
    np.ones_like(X_train_reach2),
    'o',
    color=reach2_color,
    alpha=0.7,
    markersize=6,
    markeredgecolor='white',
    markeredgewidth=0.4,
    label=f'{angles[reach_idx2]}° reach',
)
ax.set_xlim(x_min, x_max)
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 0.5, 1])
ax.set_xlabel(f'Neuron {neuron_idx} firing rate (Hz)')
ax.set_ylabel(f'Probability of {angles[reach_idx2]}° reach')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.spines[['top', 'right']].set_visible(False)
out = widgets.Output()


def render(w, b):
    sigmoid_curve.set_ydata(sigmoid(w * x_curve + b))
    prob_img.set_data(sigmoid(w * x_curve + b)[None, :])
    if w != 0:
        crossing_line.set_xdata([-b / w, -b / w])
        crossing_line.set_visible(True)
    else:
        crossing_line.set_visible(False)
    y_pred_train = (sigmoid(w * X_train + b) > 0.5).astype(int)
    y_pred_test = (sigmoid(w * X_test + b) > 0.5).astype(int)
    acc_train = accuracy(y_train, y_pred_train)
    acc_test = accuracy(y_test, y_pred_test)
    ll_train = -cross_entropy_loss(y_train, w, b, X_train)
    ll_test = -cross_entropy_loss(y_test, w, b, X_test)
    ax.set_title(
        f'w = {w:.2f},  b = {b:.1f}\n'
        f'train: acc {acc_train:.2f}, log-lik {ll_train:.2f}    |    '
        f'test: acc {acc_test:.2f}, log-lik {ll_test:.2f}'
    )
    with out:
        out.clear_output(wait=True)
        display(fig)


w_slider = widgets.FloatSlider(min=3.0, max=5.0, step=0.01, value=3.2, description='w')
b_slider = widgets.FloatSlider(min=-25, max=-20, step=0.01, value=-21, description='b')
w_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
b_slider.observe(lambda change: render(w_slider.value, b_slider.value), names='value')
render(w_slider.value, b_slider.value)
display(widgets.HBox([w_slider, b_slider]), out)

**_💬 Discussion questions:_**

1. How does the log-likelihood relate to the classification accuracy? As classification accuracy increases, what happens to the log-likelihood?
2. At what values of $w$ and $b$ does the log-likelihood reach its maximum? Are these the same values that you identified earlier?

Let's go through the same excericise, but this time, let's fix the bias term, and only adjust the weight. Then we'll also visualize the cross-entropy loss as a function of the weight parameter.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

reach1_color, reach2_color, sigmoid_color = 'tab:red', 'tab:blue', '#5b3fa0'
b = -22.0  # fixed bias term; here we vary only the weight w

# Precompute the cross-entropy loss curve over w (b fixed) ONCE.
w_vals = np.linspace(3, 5, 100)
losses = np.array([cross_entropy_loss(y_train, w, b, X_train) for w in w_vals])

x_min, x_max = X_train.min() - 0.2, X_train.max() + 0.2
x_curve = np.linspace(x_min, x_max, 200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
plt.close(fig)

# Left panel: the logistic sigmoid on the data.
# class-probability heatmap
prob_img = ax1.imshow(
    sigmoid(x_curve)[None, :],
    extent=[x_min, x_max, -0.1, 1.1],
    cmap='coolwarm_r',
    vmin=0,
    vmax=1,
    alpha=0.25,
    aspect='auto',
    origin='lower',
    zorder=0,
)
ax1.axhline(0.5, color='gray', linestyle='--', linewidth=1)
crossing_line = ax1.axvline(0, color='gray', linestyle='--', linewidth=1)
(sigmoid_curve,) = ax1.plot(x_curve, sigmoid(x_curve), color=sigmoid_color, linewidth=2.2)
ax1.plot(
    X_train_reach1,
    np.zeros_like(X_train_reach1),
    'o',
    color=reach1_color,
    alpha=0.7,
    markersize=6,
    markeredgecolor='white',
    markeredgewidth=0.4,
    label=f'{angles[reach_idx1]}° reach',
)
ax1.plot(
    X_train_reach2,
    np.ones_like(X_train_reach2),
    'o',
    color=reach2_color,
    alpha=0.7,
    markersize=6,
    markeredgecolor='white',
    markeredgewidth=0.4,
    label=f'{angles[reach_idx2]}° reach',
)
ax1.set_xlim(x_min, x_max)
ax1.set_ylim(-0.1, 1.1)
ax1.set_yticks([0, 0.5, 1])
ax1.set_xlabel(f'Neuron {neuron_idx} firing rate (Hz)')
ax1.set_ylabel(f'Probability of {angles[reach_idx2]}° reach')
ax1.legend(loc='center left', frameon=False)
ax1.spines[['top', 'right']].set_visible(False)

# Right panel: the loss curve, with a dot marking the current weight.
ax2.plot(w_vals, losses, color=sigmoid_color, linewidth=2.0)
(loss_dot,) = ax2.plot([], [], 'o', color='tab:red', markersize=9)
ax2.set_xlabel('Weight w')
ax2.set_ylabel('Cross-entropy loss')
ax2.set_title(f'Loss vs. weight (b = {b:.1f})')
ax2.spines[['top', 'right']].set_visible(False)
out = widgets.Output()


def render(w):
    sigmoid_curve.set_ydata(sigmoid(w * x_curve + b))
    prob_img.set_data(sigmoid(w * x_curve + b)[None, :])
    if w != 0:
        crossing_line.set_xdata([-b / w, -b / w])
        crossing_line.set_visible(True)
    else:
        crossing_line.set_visible(False)
    acc_train = accuracy(y_train, (sigmoid(w * X_train + b) > 0.5).astype(int))
    ll_train = -cross_entropy_loss(y_train, w, b, X_train)
    ax1.set_title(f'w = {w:.2f}, b = {b:.1f}   |   train acc {acc_train:.2f}, log-lik {ll_train:.2f}')
    loss_dot.set_data([w], [cross_entropy_loss(y_train, w, b, X_train)])
    with out:
        out.clear_output(wait=True)
        display(fig)


w_slider = widgets.FloatSlider(min=3, max=5, step=0.01, value=3.5, description='w')
w_slider.observe(lambda change: render(w_slider.value), names='value')
render(w_slider.value)
display(w_slider, out)

#### Automatically finding the optimal parameters with gradient descent

We're now ready to define an algorithm to find the optimal parameters $w$ and $b$
that minimize the loss function.

We will use the gradient descent algorithm, which iteratively updates the parameters
using the gradient of the loss function with respect to those parameters.

Intuitively, the (negative) gradient points "downhill," indicating the direction in
which the loss function decreases most rapidly.

In [ ]:
# Visualize the loss curve and the (negative) gradient at several weights.
b = -15.0  # fixed bias, chosen so the loss minimum sits mid-range
w_vals = np.linspace(0, 5, 200)
losses = np.array([cross_entropy_loss(y_train, w, b, X_train) for w in w_vals])

# Sample points where we draw the descent direction.
w_pts = np.linspace(0.5, 4.5, 9)
sigmoid_color, arrow_color = '#5b3fa0', '#d1495b'
step_scale = 0.16  # visual length of the gradient arrows

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(w_vals, losses, color=sigmoid_color, linewidth=2.4, zorder=1)
for w in w_pts:
    loss_w = cross_entropy_loss(y_train, w, b, X_train)
    grad_w = X_train.T @ (sigmoid(X_train * w + b) - y_train) / len(y_train)
    ax.plot(w, loss_w, 'o', color=sigmoid_color, markersize=5, zorder=4)
    # Arrow points in the negative-gradient (downhill) direction, scaled by |gradient|.
    ax.annotate('', xy=(w - step_scale * grad_w, loss_w), xytext=(w, loss_w),
                arrowprops=dict(arrowstyle='-|>', color=arrow_color, linewidth=1.8), zorder=3)

# Mark the minimum.
j_min = losses.argmin()
ax.axvline(w_vals[j_min], color='gray', linestyle='--', linewidth=1, zorder=0)
ax.plot(w_vals[j_min], losses[j_min], 'o', color='k', markersize=6, zorder=5)

ax.plot([], [], color=arrow_color, linewidth=1.8, label='negative gradient\n(downhill step)')
ax.legend(loc='upper right', frameon=False)
ax.set_xlabel('Weight w')
ax.set_ylabel('Cross-entropy loss')
ax.set_title(f'Loss vs. weight with gradients (b = {b:.1f})')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

The derivations for the gradient of the loss function with respect to the parameters $w$
and $b$ require fairly involved algebra and applications of the chain rule in calculus.

We therefore take the following expressions for granted. In the 1D case:
- The derivative or gradient of the loss function with respect to $w$
  is given by:
  $$\frac{\partial L}{\partial w} = \frac{\partial}{\partial w} \left( -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log\left(\sigma(w x_i + b)\right) + (1 - y_i) \log\left(1 - \sigma(w x_i + b)\right) \right] \right)$$
  $$= -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i - \sigma(w x_i + b) \right] \cdot x_i$$
- The derivative or gradient of the loss function with respect to $b$
  is given by:
  $$\frac{\partial L}{\partial b} = \frac{\partial}{\partial b} \left( -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log\left(\sigma(w x_i + b)\right) + (1 - y_i) \log\left(1 - \sigma(w x_i + b)\right) \right] \right)$$
  $$= -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i - \sigma(w x_i + b) \right]$$

Pseudo-code for the gradient descent algorithm is then as follows:
```bash
    >>> given learning rate α, and number of iterations num_iter
    >>> initialize w, b
    >>> for num_iters:
    >>>    compute: dL/dw
    >>>    compute: dL/db
    >>>    update: w = w - α ⋅ dL/dw
    >>>    update: b = b - α ⋅ dL/db
    >>> return w, b
```

#### Putting it all together: the logistic regression classifier

We now have all the tools we need to implement a logistic regression classifier.
We will create a class that implements the logistic regression algorithm, with a
similar structure to the KNN class we created earlier.

**_📝 Exercise_**

In the code below, implement the gradient descent steps for the weight $w$ and the bias $b$.

Wherever you see a `TODO` (there are **two**), attempt to fill in that line of code.

You can find the solution below.

In [ ]:
class LogisticRegression:
    def __init__(self, lr: float = 0.001, num_iter: int = 1000, w_init=None, b_init: float = 0.0):
        """
        Initialize the logistic regression model.

        Parameters
        ----------
        lr : float
            Learning rate for gradient descent. Defaults to 0.001.
        num_iter : int
            Number of iterations for gradient descent. Defaults to 1000.
        w_init : array-like, optional
            Initial weight(s). Defaults to zeros. Choosing a start near a good
            solution makes gradient descent converge in fewer iterations.
        b_init : float
            Initial bias. Defaults to 0.
        """
        self.lr = lr
        self.num_iter = num_iter
        self.w_init = w_init
        self.b_init = b_init
        self.weights = None
        self.bias = None
        self.loss = cross_entropy_loss

    def fit(self, X: np.ndarray, y: np.ndarray):
        """
        Fit the logistic regression model to the training data.

        Parameters
        ----------
        X : numpy array, shape (n, d)
            Input data, where n is the number of samples and d is the number of
            features.
        y : numpy array, shape (n,)
            Target labels (0 or 1) for each sample.

        Returns
        -------
        loss_history : list
            List of cross-entropy loss values at each iteration.
        weight_history : list of numpy arrays
            List of weight vectors at each iteration.
        bias_history : list
            List of bias values at each iteration.
        """
        # Check if X is a 1D array and reshape it to 2D if necessary
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        n, d = X.shape

        # Initialize weights and bias. Default is zeros; passing w_init / b_init
        # lets us start elsewhere (e.g. near a good solution) for faster convergence.
        self.weights = np.zeros(d) if self.w_init is None else np.array(self.w_init, dtype=float).reshape(d)
        self.bias = float(self.b_init)
        # Initialize lists to store the history of weights, bias, and loss
        loss_history = []
        weight_history = []
        bias_history = []

        # Main training loop that performs gradient descent
        for _ in range(self.num_iter):
            # Compute the class probabilities for each data point
            logits = X @ self.weights + self.bias
            class_probs = sigmoid(logits)
            # Compute the cross-entropy loss
            loss = self.loss(y, self.weights, self.bias, X)
            loss_history.append(loss)

            # Compute the gradient of the loss function with respect to weights and bias
            dw = (1 / n) * X.T @ (class_probs - y)
            db = (1 / n) * np.sum(class_probs - y)

            # Perform a gradient descent step
            self.weights = None  # TODO: Replace None. Use self.lr and dw
            self.bias = None     # TODO: Replace None. Use self.lr and db

            # Update the history of weights and bias
            weight_history.append(self.weights.copy())
            bias_history.append(self.bias)

        return loss_history, weight_history, bias_history

    def predict(self, X):
        """
        Predict the class labels for the input data using the fitted model.

        Parameters
        ----------
        X : numpy array, shape (n, d)
            Input data, where n is the number of samples and d is the number of
            features.

        Returns
        -------
        class_pred : list
            List of predicted class labels (0 or 1) for each sample.
        """
        # Check if X is a 1D array and reshape it to 2D if necessary
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        # Compute the class probabilities
        logits = X @ self.weights + self.bias
        class_probs = sigmoid(logits)
        # Convert probabilities to class predictions
        class_pred = [0 if class_prob <= 0.5 else 1 for class_prob in class_probs]
        return class_pred

In [ ]:
#@title Double click to see solution {display-mode: "form" }
class LogisticRegression:
    def __init__(self, lr: float = 0.001, num_iter: int = 1000, w_init=None, b_init: float = 0.0):
        """
        Initialize the logistic regression model.

        Parameters
        ----------
        lr : float
            Learning rate for gradient descent. Defaults to 0.001.
        num_iter : int
            Number of iterations for gradient descent. Defaults to 1000.
        w_init : array-like, optional
            Initial weight(s). Defaults to zeros. Choosing a start near a good
            solution makes gradient descent converge in fewer iterations.
        b_init : float
            Initial bias. Defaults to 0.
        """
        self.lr = lr
        self.num_iter = num_iter
        self.w_init = w_init
        self.b_init = b_init
        self.weights = None
        self.bias = None
        self.loss = cross_entropy_loss

    def fit(self, X: np.ndarray, y: np.ndarray):
        """
        Fit the logistic regression model to the training data.

        Parameters
        ----------
        X : numpy array, shape (n, d)
            Input data, where n is the number of samples and d is the number of
            features.
        y : numpy array, shape (n,)
            Target labels (0 or 1) for each sample.

        Returns
        -------
        loss_history : list
            List of cross-entropy loss values at each iteration.
        weight_history : list of numpy arrays
            List of weight vectors at each iteration.
        bias_history : list
            List of bias values at each iteration.
        """
        # Check if X is a 1D array and reshape it to 2D if necessary
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        n, d = X.shape

        # Initialize weights and bias. Default is zeros; passing w_init / b_init
        # lets us start elsewhere (e.g. near a good solution) for faster convergence.
        self.weights = np.zeros(d) if self.w_init is None else np.array(self.w_init, dtype=float).reshape(d)
        self.bias = float(self.b_init)
        # Initialize lists to store the history of weights, bias, and loss
        loss_history = []
        weight_history = []
        bias_history = []

        # Main training loop that performs gradient descent
        for _ in range(self.num_iter):
            # Compute the class probabilities for each data point
            logits = X @ self.weights + self.bias
            class_probs = sigmoid(logits)
            # Compute the cross-entropy loss
            loss = self.loss(y, self.weights, self.bias, X)
            loss_history.append(loss)

            # Compute the gradient of the loss function with respect to weights and bias
            dw = (1 / n) * X.T @ (class_probs - y)
            db = (1 / n) * np.sum(class_probs - y)

            # Perform a gradient descent step
            self.weights -= self.lr * dw  # SOLUTION
            self.bias -= self.lr * db     # SOLUTION

            # Update the history of weights and bias
            weight_history.append(self.weights.copy())
            bias_history.append(self.bias)

        return loss_history, weight_history, bias_history

    def predict(self, X):
        """
        Predict the class labels for the input data using the fitted model.

        Parameters
        ----------
        X : numpy array, shape (n, d)
            Input data, where n is the number of samples and d is the number of
            features.

        Returns
        -------
        class_pred : list
            List of predicted class labels (0 or 1) for each sample.
        """
        # Check if X is a 1D array and reshape it to 2D if necessary
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        # Compute the class probabilities
        logits = X @ self.weights + self.bias
        class_probs = sigmoid(logits)
        # Convert probabilities to class predictions
        class_pred = [0 if class_prob <= 0.5 else 1 for class_prob in class_probs]
        return class_pred

Now let's create a LogisticRegression instance and fit it to our training data.

In [ ]:
log_reg = LogisticRegression(lr=0.002, w_init=3.2, b_init=-21.0, num_iter=250)
loss_history, weight_history, bias_history = log_reg.fit(X_train, y_train)
# Let's predict the labels for the training and test data
y_pred_train = log_reg.predict(X_train)
y_pred_test = log_reg.predict(X_test)
# Compute the accuracy for the training and test data
acc_train = accuracy(y_train, y_pred_train)
acc_test = accuracy(y_test, y_pred_test)
# Print the learned weights and bias
print(f'Learned Weights: {log_reg.weights}')
print(f'Learned Bias: {log_reg.bias}')
# Print the accuracies
print(f'Train Accuracy: {acc_train:.3f}')
print(f'Test Accuracy: {acc_test:.3f}')

# Plot the loss, weight, and bias histories over training.
fig, (ax_loss, ax_w, ax_b) = plt.subplots(1, 3, figsize=(12, 3.5))
fig.subplots_adjust(wspace=0.32)

ax_loss.plot(loss_history, color='#5b3fa0', linewidth=2)
ax_loss.set_xlabel('iteration')
ax_loss.set_ylabel('cross-entropy loss')
ax_loss.set_title('Loss')

ax_w.plot(weight_history, color='tab:blue', linewidth=2)
ax_w.set_xlabel('iteration')
ax_w.set_ylabel('weight w')
ax_w.set_title('Weight')

ax_b.plot(bias_history, color='tab:green', linewidth=2)
ax_b.set_xlabel('iteration')
ax_b.set_ylabel('bias b')
ax_b.set_title('Bias')

for ax in (ax_loss, ax_w, ax_b):
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**_💬 Discussion questions:_**

1. How do the parameters learned by this logistic regression classifier compare to the parameters you found earlier by hand?
2. How does the performance of this classifier compare to the KNN classifier?

#### We can just as well apply our logistic regression model to the 2D data

In [ ]:
#@title Back to our 2D data {display-mode: "form" }
neuron_idxs = demo_neuron_pair  # The neurons we'll use for binary classification
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, neuron_idxs].T
X_reach2 = Xplan_angles[reach_idx2, :, neuron_idxs].T

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])   # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

Create a LogisticRegression instance and fit it to our training data

In [ ]:
log_reg = LogisticRegression(lr=0.001, num_iter=250, w_init=[0.75, -0.55], b_init=-1.5)
loss_history, weight_history, bias_history = log_reg.fit(X_train, y_train)
# Let's predict the labels for the training and test data
y_pred_train = log_reg.predict(X_train)
y_pred_test = log_reg.predict(X_test)
# Compute the accuracy for the training and test data
acc_train = accuracy(y_train, y_pred_train)
acc_test = accuracy(y_test, y_pred_test)
# Print the accuracies
print(f'Train Accuracy: {acc_train:.3f}')
print(f'Test Accuracy: {acc_test:.3f}')

# Plot the loss, weight, and bias histories over training.
fig, (ax_loss, ax_w, ax_b) = plt.subplots(1, 3, figsize=(12, 3.5))
fig.subplots_adjust(wspace=0.32)

ax_loss.plot(loss_history, color='#5b3fa0', linewidth=2)
ax_loss.set_xlabel('iteration')
ax_loss.set_ylabel('cross-entropy loss')
ax_loss.set_title('Loss')

ax_w.plot(weight_history, linewidth=1.5)  # one line per neuron
ax_w.set_xlabel('iteration')
ax_w.set_ylabel('weights')
ax_w.set_title('Weights (one per neuron)')

ax_b.plot(bias_history, color='tab:green', linewidth=2)
ax_b.set_xlabel('iteration')
ax_b.set_ylabel('bias b')
ax_b.set_title('Bias')

for ax in (ax_loss, ax_w, ax_b):
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**_💬 Discussion questions:_**

1. How do the parameters learned by this logistic regression classifier compare to the
   parameters you found earlier by hand?
2. How does the performance of this classifier compare to the KNN classifier?

**_📝 Exercises_**

1. Rerun the 2D logistic regression problem, but try different values for the learning rate and number of iterations. How does the performance of the classifier change?
2. Rerun the above analyses, but try different pairs of reach angles to classify. Which pairs result in the best classification performance, and why? Which pairs result in the the worst classification performance, and why?

## *Bonus 2*. Population decoding <a name="population-decoding"></a>

We have seen how to decode the reach angle from the activity of a single neuron, or
from the activity of two neurons. But what if we have many more neurons? How do we
decode the reach angle from the activity of a population of neurons?

It turns out that we have all the tools we need to do this, with either the KNN
classifier or the logistic regression classifier.

In [ ]:
# Prepare the data. This time, we'll use all of the neurons in our recording!
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, :]
X_reach2 = Xplan_angles[reach_idx2, :, :]

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])   # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

### Population decoding with logistic regression

In [ ]:
# Create a LogisticRegression instance and fit it to our training data
log_reg = LogisticRegression(lr=0.005, num_iter=2500)
loss_history, weight_history, bias_history = log_reg.fit(X_train, y_train)
# Let's predict the labels for the training and test data
y_pred_train = log_reg.predict(X_train)
y_pred_test = log_reg.predict(X_test)
# Compute the accuracy for the training and test data
acc_train = accuracy(y_train, y_pred_train)
acc_test = accuracy(y_test, y_pred_test)
# Print the accuracies
print(f'Train Accuracy: {acc_train:.3f}')
print(f'Test Accuracy: {acc_test:.3f}')

# Plot the loss, weight, and bias histories over training.
fig, (ax_loss, ax_w, ax_b) = plt.subplots(1, 3, figsize=(12, 3.5))
fig.subplots_adjust(wspace=0.32)

ax_loss.plot(loss_history, color='#5b3fa0', linewidth=2)
ax_loss.set_xlabel('iteration')
ax_loss.set_ylabel('cross-entropy loss')
ax_loss.set_title('Loss')

ax_w.plot(weight_history, linewidth=1.5)  # one line per neuron
ax_w.set_xlabel('iteration')
ax_w.set_ylabel('weights')
ax_w.set_title('Weights (one per neuron)')

ax_b.plot(bias_history, color='tab:green', linewidth=2)
ax_b.set_xlabel('iteration')
ax_b.set_ylabel('bias b')
ax_b.set_title('Bias')

for ax in (ax_loss, ax_w, ax_b):
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### Population decoding with Nearest Neighbors

In [ ]:
knn_ND = KNNBase(k=21)
knn_ND.distance = euclidean_distance

# Fit the KNN model
knn_ND.fit(X_train, y_train)

# Predict the labels for the train data
y_pred_train = knn_ND.predict_batch(X_train)

# Predict the labels for the test data
y_pred_test = knn_ND.predict_batch(X_test)

# Train accuracy
acc_train = accuracy(y_train, y_pred_train)
print(f'Train accuracy: {acc_train:.3f}')

# Test accuracy
acc_test = accuracy(y_test, y_pred_test)
print(f'Test accuracy: {acc_test:.3f}')

**_📝 Exercises_**

1. Rerun the above analyses, but try different pairs of reach angles to classify.
2. Across the pairs, try different settings for the KNN and Logistic Regression classifiers. How do the results change? Which classifier performs better? Is that surprising? Why might one classifier be consistently performing better than the other?

## *Bonus 3*. Combining classification and PCA <a name="class-pca"></a>

We can combine classification and PCA as follows:
1. First, we apply PCA to the neural data to reduce its dimensionality.
2. Then, we use the reduced-dimensionality data as input to a classification algorithm,
   such as KNN or logistic regression.

In [ ]:
# Prepare the data. This time, we'll use all of the neurons in our recording!
reach_idx1 = 0  # First reach angle
reach_idx2 = 1  # Second reach angle

# Extract the data for the two angles
X_reach1 = Xplan_angles[reach_idx1, :, :]
X_reach2 = Xplan_angles[reach_idx2, :, :]

# Let's create a label vector for the two angles
y_reach1 = np.zeros(X_reach1.shape[0])  # Label for the first angle
y_reach2 = np.ones(X_reach2.shape[0])   # Label for the second angle

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train_reach1 = X_reach1[train_idxs]
X_train_reach2 = X_reach2[train_idxs]
y_train_reach1 = y_reach1[train_idxs]
y_train_reach2 = y_reach2[train_idxs]
X_train = np.concatenate((X_train_reach1, X_train_reach2))
y_train = np.concatenate((y_train_reach1, y_train_reach2))

# Test data
X_test_reach1 = X_reach1[test_idxs]
X_test_reach2 = X_reach2[test_idxs]
y_test_reach1 = y_reach1[test_idxs]
y_test_reach2 = y_reach2[test_idxs]
X_test = np.concatenate((X_test_reach1, X_test_reach2))
y_test = np.concatenate((y_test_reach1, y_test_reach2))

Apply PCA to these two reach angles and visualize 2D projections

In [ ]:
# Apply PCA to these two reach angles and visualize the 2D projections

# First find the mean over trials
mu = np.mean(X_train, axis=0)

# Now compute the data covariance
X_train_centered = X_train - mu                      # Don't forget to center your data first!
cov = (1/n) * X_train_centered.T @ X_train_centered  # Now you can compute the covariance

# Perform eigendecomposition of the data covariance
D, U = np.linalg.eig(cov)  # D = vector of eigenvalues, U = matrix of eigenvectors

# Make sure the eigenvalues are sorted (in descending order)
idx = np.argsort(D)[::-1]
D    = D[idx]

# Arrange the eigenvectors according to the magnitude of the eigenvalues
U = U[:, idx]

# Project the data onto the first two principal components
X_train_pca = X_train_centered @ U[:, :2]

# Visualize the PCA projection.
reach1_color, reach2_color = 'tab:red', 'tab:blue'
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(
    X_train_pca[y_train == 0, 0],
    X_train_pca[y_train == 0, 1],
    color=reach1_color,
    alpha=0.6,
    s=36,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx1]}° reach',
)
ax.scatter(
    X_train_pca[y_train == 1, 0],
    X_train_pca[y_train == 1, 1],
    color=reach2_color,
    alpha=0.6,
    s=36,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx2]}° reach',
)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title(f'PCA projection of {angles[reach_idx1]}° and {angles[reach_idx2]}° reaches')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

Classify the PCA projections with Logistic Regression

In [ ]:
# Classify the PCA projections with Logistic Regression
log_reg = LogisticRegression(lr=0.001, num_iter=1000)
loss_history, weight_history, bias_history = log_reg.fit(X_train_pca, y_train)
y_pred_train = log_reg.predict(X_train_pca)
acc_train = accuracy(y_train, y_pred_train)

# Project the held-out test data with the SAME PCA (mean mu, components U from above).
X_test_pca = (X_test - mu) @ U[:, :2]
y_pred_test = log_reg.predict(X_test_pca)
acc_test = accuracy(y_test, y_pred_test)
print(f'Train Accuracy: {acc_train:.3f}')
print(f'Test Accuracy: {acc_test:.3f}')

# Visualize the learned decision boundary and class-probability map in PC space.
reach1_color, reach2_color = 'tab:red', 'tab:blue'
x_min, x_max = X_train_pca[:, 0].min() - 0.5, X_train_pca[:, 0].max() + 0.5
y_min, y_max = X_train_pca[:, 1].min() - 0.5, X_train_pca[:, 1].max() + 0.5
gx, gy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
prob = sigmoid(np.column_stack([gx.ravel(), gy.ravel()]) @ log_reg.weights + log_reg.bias).reshape(gx.shape)

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(
    prob,
    origin='lower',
    extent=[x_min, x_max, y_min, y_max],
    cmap='coolwarm_r',
    vmin=0,
    vmax=1,
    alpha=0.75,
    aspect='auto',
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(f'P({angles[reach_idx2]}° reach)')
ax.contour(gx, gy, prob, levels=[0.5], colors='0.2', linewidths=1.2)
ax.scatter(
    X_train_pca[y_train == 0, 0],
    X_train_pca[y_train == 0, 1],
    color=reach1_color,
    alpha=0.7,
    s=32,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx1]}° reach',
)
ax.scatter(
    X_train_pca[y_train == 1, 0],
    X_train_pca[y_train == 1, 1],
    color=reach2_color,
    alpha=0.7,
    s=32,
    edgecolor='white',
    linewidth=0.4,
    label=f'{angles[reach_idx2]}° reach',
)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('Logistic decision boundary in PC space')
ax.legend(loc='upper left', framealpha=0.8, edgecolor='none')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## *Bonus 4*. Multiclass classification <a name="multiclass"></a>

The code we wrote for the KNN classifier is readily applicable to multiclass
classification. We can simply extend the majority vote rule to consider all classes,
and then classify a test point based on the class with the most votes.

In [ ]:
# Let's create labels for all angles
labels = np.arange(len(angles))
y = np.concatenate([np.full(num_reaches_per_angle, label) for label in labels])

# Let's also randomly split the data into training and test sets.
# We'll keep the same number of trials for each angle.
n_train = int(train_frac * num_reaches_per_angle)  # training fraction (from the dataset)
# Seed the random number generator for reproducibility
np.random.seed(split_seed)
train_idxs = np.random.choice(num_reaches_per_angle, n_train, replace=False)
# This function just returns the indices that are not already in the training set
test_idxs = np.setdiff1d(np.arange(num_reaches_per_angle), train_idxs)

# Train data
X_train = Xplan_angles[:, train_idxs, :].reshape(-1, Xplan_angles.shape[2])
y = y.reshape(num_angles, -1)
y_train = y[:, train_idxs]
y_train = y_train.reshape(-1)

# Test data
X_test = Xplan_angles[:, test_idxs, :].reshape(-1, Xplan_angles.shape[2])
y_test = y[:, test_idxs]
y_test = y_test.reshape(-1)

Classify this 8-reach data with a KNN classifier

In [ ]:
# Classify this 8-reach data with a KNN classifier
knn_ND = KNNBase(k=21)
knn_ND.distance = euclidean_distance
knn_ND.fit(X_train, y_train)

# Predict the labels for the train data
y_pred_train = knn_ND.predict_batch(X_train)

# Predict the labels for the test data
y_pred_test = knn_ND.predict_batch(X_test)

# Train accuracy
acc_train = accuracy(y_train, y_pred_train)
print(f'Train accuracy: {acc_train:.3f}')

# Test accuracy
acc_test = accuracy(y_test, y_pred_test)
print(f'Test accuracy: {acc_test:.3f}')

## *Bonus 5*. Further reading <a name="further-reading"></a>

Scikit-Learn's methods:
1. [Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html#nearest-neighbors-classification)
2. [Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)

Modern decoding applications:
1. [High-performance brain-to-text communication via handwriting](https://doi.org/10.1038/s41586-021-03506-2)
2. [A generic noninvasive neuromotor interface for human-computer interaction](https://doi.org/10.1038/s41586-025-09255-w)

Textbook chapters:
1. [Pattern Recognition and Machine Learning, Chapter 4](https://www.microsoft.com/en-us/research/wp-content/uploads/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf)
2. [The Elements of Statistical Learning, Chapter 4](https://hastie.su.domains/ElemStatLearn/)

## **About the author: Evren Gokcen**

* Research Scientist at Bosch Research, Multi-Modal Sensing AI Group, Pittsburgh, PA, USA
* My research has been heavily interdisciplinary, spanning AI, machine learning, signal processing, and optimal control, with applications including computational neuroscience, brain-computer interfaces, and wireless sensing.
* Feel free to contact me: egokcen@cmu.edu
* You can also find me on [GitHub](https://github.com/egokcen), [LinkedIn](https://www.linkedin.com/in/evrengokcen/), and [Google Scholar](https://scholar.google.com/citations?hl=en&user=fal6YjcAAAAJ)